# Import Thư viện

In [67]:
import pandas as pd
import matplotlib.pyplot  as plt
import numpy              as np
import plotly.express       as px
import pydotplus         as pdp
import seaborn as sns
from scipy import stats
from sklearn import tree
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.metrics          import mean_absolute_error, mean_squared_error
from IPython.display         import Image
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split
from sklearn.metrics          import accuracy_score, confusion_matrix
from sklearn.metrics          import auc,roc_curve
from sklearn.metrics          import precision_score, recall_score, f1_score
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report
from sklearn.model_selection import RandomizedSearchCV
from sklearn.preprocessing import PowerTransformer
from scipy.stats.mstats import winsorize

# Import Data

In [68]:
df3 = pd.read_csv('./data/paris_weekdays.csv')
df4 = pd.read_csv('./data/paris_weekends.csv')

df3['weekends'] = 0
df4['weekends'] = 1

d = [df3, df4]
df = pd.concat(d, axis= 0)
df = df.rename(columns= {df.columns[0]: 'ID'})
df.drop(['ID', 'attr_index_norm', 'rest_index_norm'], axis=1, inplace=True)
df

,realSum,room_type,room_shared,room_private,person_capacity,host_is_superhost,multi,biz,cleanliness_rating,guest_satisfaction_overall,bedrooms,dist,metro_dist,attr_index,rest_index,lng,lat,weekends
0,296.159940,Private room,False,True,2.0,True,0,0,10.0,97.0,1,0.699821,0.193709,518.478947,1218.662228,2.35385,48.86282,0
1,288.237487,Private room,False,True,2.0,True,0,0,10.0,97.0,1,2.100005,0.107221,873.216962,1000.543327,2.32436,48.85902,0
2,211.343089,Private room,False,True,2.0,False,0,0,10.0,94.0,1,3.302325,0.234724,444.556077,902.854467,2.31714,48.87475,0
3,298.956100,Entire home/apt,False,False,2.0,False,0,1,9.0,91.0,1,0.547567,0.195997,542.142014,1199.184166,2.35600,48.86100,0
4,247.926181,Entire home/apt,False,False,4.0,False,0,0,7.0,82.0,1,1.197921,0.103573,406.928958,1070.775497,2.35915,48.86648,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3553,295.460900,Entire home/apt,False,False,4.0,False,0,0,9.0,80.0,1,3.660389,0.168146,209.752453,540.326583,2.38051,48.88393,1
3554,232.081275,Entire home/apt,False,False,4.0,False,0,0,10.0,98.0,1,3.558813,0.154703,185.486701,474.351813,2.40050,48.85093,1
3555,223.925809,Entire home/apt,False,False,2.0,False,1,0,9.0,89.0,1,4.205205,0.253029,172.658919,406.585935,2.40100,48.87700,1
3556,200.857489,Entire home/apt,False,False,2.0,True,0,0,9.0,93.0,1,2.891214,0.240674,235.167925,602.451672,2.38200,48.87400,1


# Tổng quan bộ dữ liệu

# Chuẩn bị dữ liệu

In [69]:
continuous_cols = ['realSum', 'guest_satisfaction_overall', 
                 'dist','metro_dist', 'attr_index','rest_index', 'lng', 'lat']
discrete_cols = ['person_capacity', 'cleanliness_rating', 'bedrooms' ]
binary_cols = ['room_shared', 'room_private', 'host_is_superhost', 'multi', 'biz', 'weekends']
nominal_cols = ['room_type']
bool_cols = ['room_shared', 'room_private', 'host_is_superhost']

num_cols = continuous_cols + discrete_cols

# Tiền xử lý

### Chung cho cả Classification và Regression

In [70]:
df['x'] = np.cos(np.radians(df['lat'])) * np.cos(np.radians(df['lng']))
df['y'] = np.cos(np.radians(df['lat'])) * np.sin(np.radians(df['lng']))
df['z'] = np.sin(np.radians(df['lat']))

df = df.drop(['lng', 'lat'], axis= 1)

In [71]:
encoder_1 = LabelEncoder()
df['room_type'] =  encoder_1.fit_transform(df['room_type'])

df[bool_cols] = df[bool_cols].astype(int)

In [72]:
df.corr()['realSum'].sort_values(ascending=False)

realSum                       1.000000
person_capacity               0.395620
bedrooms                      0.353561
attr_index                    0.252642
rest_index                    0.212614
biz                           0.104630
cleanliness_rating            0.068471
guest_satisfaction_overall    0.058530
x                             0.045607
host_is_superhost             0.022569
multi                         0.018087
weekends                     -0.017729
metro_dist                   -0.026076
z                            -0.036172
room_shared                  -0.086654
dist                         -0.099366
y                            -0.139742
room_private                 -0.153378
room_type                    -0.181257
Name: realSum, dtype: float64

In [73]:
df_clf = df.copy()
df_reg = df.copy()

df.shape

(6688, 19)

### Utility

In [74]:
def auto_transform_skewed(df, numeric_cols, skew_threshold=0.5):
    df_trans = df.copy()
    transform_summary = {}

    for col in numeric_cols:
        skew = df[col].skew()
        method = None

        if abs(skew) < skew_threshold:
            continue

        if skew > skew_threshold:
            if (df[col] > 0).all():
                df_trans[col], _ = stats.boxcox(df[col])
                method = 'Box-Cox'
            else:
                pt = PowerTransformer(method='yeo-johnson')
                df_trans[col] = pt.fit_transform(df[[col]])
                method = 'Yeo-Johnson'

        elif skew < -skew_threshold:
            flipped = -df[col] + df[col].max() + 1
            pt = PowerTransformer(method='yeo-johnson')
            df_trans[col] = -pt.fit_transform(flipped.to_frame())
            method = 'Power transform (left skew)'

        transform_summary[col] = {'skew_before': skew, 'method': method}

    return df_trans, pd.DataFrame(transform_summary).T




In [75]:
# Tính trung vị của giá
median_price = df_clf['realSum'].median()

# Hàm phân loại 2 nhóm giá
def classify_price(x):
    if x <= median_price:
        return 'Thấp'
    else:
        return 'Cao'

In [76]:
# Tạo cột nhãn
df_clf['price_group'] = df_clf['realSum'].apply(classify_price)

encoder = OrdinalEncoder()
df_clf[['room_type']] = encoder.fit_transform(df_clf[['room_type']])

In [77]:
X_reg = df_reg.drop('realSum', axis=1, inplace=False)
y_reg = df_reg['realSum']

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42)

X_clf = df_clf.drop(['realSum', 'price_group'], axis=1)
y_clf = df_clf['price_group']

X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X_clf, y_clf, test_size=0.2, random_state=42, stratify=y_clf)

In [78]:
def tune_and_prune(model, X_train, y_train, X_test, y_test, task='reg'):
    """task='reg' hoặc 'clf'"""
    if task == 'reg':
        scoring = 'r2'
    else:
        scoring = 'accuracy'

    param_grid = {
        'max_depth': [4, 6, 8, 10, None],
        'min_samples_leaf': [5, 10, 20, 30],
        'ccp_alpha': [0.0]  # tạm để 0, sẽ tìm riêng
    }

    grid = GridSearchCV(model, param_grid, cv=5, scoring=scoring, n_jobs=-1)
    grid.fit(X_train, y_train)
    best_model = grid.best_estimator_

    # ---- Cost Complexity Pruning ----
    path = best_model.cost_complexity_pruning_path(X_train, y_train)
    alphas = path.ccp_alphas[:-1]

    train_scores, test_scores = [], []
    for alpha in alphas:
        m = type(best_model)(**{**grid.best_params_, 'ccp_alpha': alpha})
        m.fit(X_train, y_train)
        if task == 'reg':
            train_scores.append(m.score(X_train, y_train))
            test_scores.append(m.score(X_test, y_test))
        else:
            train_scores.append(accuracy_score(y_train, m.predict(X_train)))
            test_scores.append(accuracy_score(y_test, m.predict(X_test)))

    best_alpha = alphas[np.argmax(test_scores)]

    pruned_model = type(best_model)(**{**grid.best_params_, 'ccp_alpha': best_alpha})
    pruned_model.fit(X_train, y_train)

    return pruned_model, best_alpha, max(test_scores)


In [79]:
# Classification
clf = DecisionTreeClassifier(random_state=42)
best_clf, alpha_clf, score_clf = tune_and_prune(
    clf, X_train_clf, y_train_clf, X_test_clf, y_test_clf, task='clf')

In [80]:
print(f"🌳 Classification: best alpha = {alpha_clf:.5f}, Accuracy test = {score_clf:.3f}")

🌳 Classification: best alpha = 0.00051, Accuracy test = 0.817


In [81]:
# Regression
reg = DecisionTreeRegressor(random_state=42)
best_reg, alpha_reg, score_reg = tune_and_prune(
    reg, X_train_reg, y_train_reg, X_test_reg, y_test_reg, task='reg')


In [82]:
print(f"🌳 Regression: best alpha = {alpha_reg:.5f}, R² test = {score_reg:.3f}")

🌳 Regression: best alpha = 6.02674, R² test = 0.476
